# WEEK5 — RAG Live (Student Version)

A compact student-friendly notebook with only headings and hint-only code cells.

## Step 1 — Setup

In [2]:
# Hint: load environment variables (e.g., OPENAI_API_KEY)
# Hint: import necessary libraries: langchain/text_splitters, vectorstores, openai client, numpy
# Hint: ensure your Python kernel has required packages installed

from pathlib import Path # Accessing files and folders easily 
import os # Operating System
from dotenv import load_dotenv # loads variables from the .env file

load_dotenv()

print('Openai api key present:', bool(os.getenv('OPENAI_API_KEY')))

print('PDF files in the current folder:', [p.name for p in Path.cwd().glob('*.pdf')])

Openai api key present: True
PDF files in the current folder: ['HDFC-Life-Group-Term-Life-Policy.pdf', 'HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf']


## Step 2 — Load PDF documents

In [6]:
# Hint: list PDF files in the data folder
# Hint: use a PDF loader to extract text into a list of documents
# Hint: verify you have 3 PDF files and show filenames to students
from langchain_community.document_loaders import PyPDFLoader

pdf_files = sorted(Path.cwd().glob('*.pdf'))

docs = []

for i in pdf_files:
    docs.extend(PyPDFLoader(str(i)).load())  ## Convert the PDF into documents(pages)

print("Loaded Pages", len(docs))


Loaded Pages 101


In [7]:
docs[0]

Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2023-08-24T19:47:11+05:30', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'author': 'Atul Bhatia', 'moddate': '2023-08-24T19:47:11+05:30', 'source': 'd:\\Mentoring\\learwithsarvesh\\AI Engineer Ready\\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\\Live\\HDFC-Life-Group-Term-Life-Policy.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1'}, page_content='F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        \n \n \n \n \n \n   HDFC Life Group Term Life \n \nOF \n \n \n«OWNERNAME» \n \n \n \n \n \n  \nBased on the Proposal and the declarations and \nany \nstatement made or referred to therein, \nWe will pay the Benefits mentioned in this Policy \nsubject to the terms and conditions contained \nherein \n \n \n \n \n \n \n<< Designation of the Authorised Signatory >>')

In [5]:
pdf_files

[WindowsPath('d:/Mentoring/learwithsarvesh/AI Engineer Ready/WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)/Live/HDFC-Life-Group-Term-Life-Policy.pdf'),
 WindowsPath('d:/Mentoring/learwithsarvesh/AI Engineer Ready/WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)/Live/HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf'),
 WindowsPath('d:/Mentoring/learwithsarvesh/AI Engineer Ready/WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)/Live/HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf')]

## Step 3 — Chunking (brief)

Context Window: Is the amount of information an AI model can "see" or "remember" at once while generating a response;

- Chunking: Breaking large text into smaller, meaningful pieces (chunks)
Why do we do this?
1. LLMs cannot process very large documents at once
2. Embeddings work best when the text is focussed and semantically tight

### Without Chunking:
1. Large Docuemnts
2. Hard to search
3. Token cost is higher
4. Poor accuracy/ Emdedding Poor

### With Chunking:
1. Small pieces
2. Easy Retrieval
3. Better matching
4. Lower Cost

In [ ]:
# Hint: choose a chunking strategy (fixed chars / sentences / token-based)
# Hint: show students the chosen chunk size and overlap
# Hint: print a single example chunk for verification

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,   
    chunk_overlap = 150
)

chunks = splitter.split_documents(docs)

print("Chunks created", len(chunks)) # If more chunks -> It has more granualr understanding

Chunks created 324


## Step 4 — Embeddings & Vector Store

Embeddings: Converting texts into numbers that capture meaning  (Semantics)

We need to search based on meaning, not keywords

In [24]:
# Hint: pick an embedding source (OpenAI or local SBERT)
# Hint: embed a small set of chunks and show vector dimension
# Hint: if using FAISS/Chroma, mention dtype=float32 requirement

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = 'rag_chroma_store1'
)

retriever = vectorstore.as_retriever(search_kwargs = {'k' : 5})

print('Vector Store is ready', len(chunks), 'chunks')


Vector Store is ready 324 chunks


- Small - ~350 D
- Medium - ~750 D
- Large - ~3000 D

## Step 5 — Retriever (simple demo)

Cosine Similarity -> CS measures how similar two vectors are based on their angle

- Not Distance
- Not magnitude
- Only direction (angle)

CS -> Core idea -> Find the meaning

- Same Direction -> Very Similar
- Opposite Direction -> Very Different
- 90 degrees -> Unrelated

- 1 -> Exactly the same meaning
- 0.5 -> Somewhat related
- 0 -> No relation
- -1 -> Opposite Meaning

In [ ]:
context_docs = retriever.invoke(question)

context_text = '\n\n'.join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in context_docs
        )

[Document(metadata={'total_pages': 30, 'creationdate': '2023-08-24T19:47:11+05:30', 'author': 'Atul Bhatia', 'source': 'd:\\Mentoring\\learwithsarvesh\\AI Engineer Ready\\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\\Live\\HDFC-Life-Group-Term-Life-Policy.pdf', 'page': 1, 'creator': 'Microsoft® Office Word 2007', 'moddate': '2023-08-24T19:47:11+05:30', 'page_label': '2', 'producer': 'Microsoft® Office Word 2007', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions'}, page_content='\uf0fc Premium Receipt   :  Acknowledgement of the first Premium paid by you \n\uf0fc Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                                                               \n                                                    Insurance \n\uf0fc Service Options  :  Wide range of Policy servicing options that you can Benefit \nfrom \n \nWe request you to carefully go through the information given in this document. You ar

In [35]:
# Hint: query the vectorstore for top-k results
# Hint: show retrieved chunk sources and short previews
# Hint: encourage students to try different k values

from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0) # Create a chat model used to generate the answer

questions = ["What are the benefits that is described in the document",
            "What is the term of my policy"]

#print("Question:", question)
for question in questions:
    context_docs = retriever.invoke(question)

    context_text = '\n\n'.join(
            f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
            for doc in context_docs
            )

    messages = [
        SystemMessage(content= "You are a helpful tutor. Answer only the provided context and keep the answer clear and student-friendly"),
        HumanMessage(content = f"Context:\n{context_text}\n\nQuestion: {question} \n\nAnswer with short citations to the source file names.")
    ]

    response = llm.invoke(messages)

    # for doc in context_docs:
    #     source_name = doc.metadata.get("source", "unknown")

    print(response.content)

The benefits described in the document include:

1. **Premium Receipt**: Acknowledgment of the first premium paid (HDFC-Life-Group-Term-Life-Policy.pdf).
2. **Terms & Conditions**: Detailed terms of the policy contract (HDFC-Life-Group-Term-Life-Policy.pdf).
3. **Service Options**: A wide range of policy servicing options available (HDFC-Life-Group-Term-Life-Policy.pdf).
4. **Benefits Payable**: All benefits are payable to the policyholder, and if assigned, to the assignee under absolute assignment (HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf). 

These benefits are essential for understanding the coverage and services provided by the policy.
The term of your policy is defined as the "Policy Year," which is a period of twelve (12) consecutive months starting from the Policy Commencement Date and ending on the day immediately preceding the following Policy anniversary date. Each subsequent period of twelve (12) consecutive months thereafter is also consi

In [ ]:
# Hint: query the vectorstore for top-k results
# Hint: show retrieved chunk sources and short previews
# Hint: encourage students to try different k values

from langchain_core.messages import SystemMessage, HumanMessage

test_questions = [
"What are the benefits that is described in the document",
"What is policy duration?"
]

for question in test_questions:
    print("Question:", question)
    context_docs = retriever.invoke(question)

    context_text = '\n\n'.join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in context_docs
        )




In [20]:
context_docs

[Document(metadata={'creator': 'Microsoft® Office Word 2007', 'source': 'd:\\Mentoring\\learwithsarvesh\\AI Engineer Ready\\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\\Live\\HDFC-Life-Group-Term-Life-Policy.pdf', 'page_label': '2', 'total_pages': 30, 'author': 'Atul Bhatia', 'page': 1, 'producer': 'Microsoft® Office Word 2007', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'creationdate': '2023-08-24T19:47:11+05:30', 'moddate': '2023-08-24T19:47:11+05:30'}, page_content='\uf0fc Premium Receipt   :  Acknowledgement of the first Premium paid by you \n\uf0fc Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                                                               \n                                                    Insurance \n\uf0fc Service Options  :  Wide range of Policy servicing options that you can Benefit \nfrom \n \nWe request you to carefully go through the information given in this document. You ar

# Class Exercises — Try in class

### Exercise 1 — Chunking experiment
**Task:** Change chunk size and overlap, rebuild the vector store, and compare retrieved chunks for a sample question. Try at least two settings (e.g., chunk_size=800, overlap=200 and chunk_size=400, overlap=100) and note differences in which chunks are retrieved.

### Exercise 2 — Hybrid tuning (lexical + dense)
**Task:** Implement a simple fusion of TF-IDF scores and dense scores (alpha-weighted) and test alpha values [0.2, 0.5, 0.8]. Report which alpha works best for 3 sample queries.


### Exercise 3 — Re-ranking and final answer quality
**Task:** Take the top-5 candidates from your retriever and re-rank them using a Cross-Encoder (or a lightweight dense dot-product re-ranker). Compare the final answer quality (before/after re-rank) for a sample question.


## Step 6 — Generate Answer (RAG)

In [ ]:
# Hint: build a prompt that includes retrieved context + user question
# Hint: call your LLM with a short system instruction (student-friendly)
# Hint: print a concise answer and cite source filenames

## Review & Exercises

- Try adjusting chunk size and re-run retrieval
- Compare OpenAI embeddings vs. a local embedding proxy

## Homework Questions

1. **Build a small RAG app**: Using the provided PDFs, build a simple retrieval-augmented chatbot (CLI or notebook) that answers policy-related questions and includes source citations. Submit a short demo video (2-3 minutes) and the code.

2. **Evaluation metrics**: For 10 sample queries, compute precision@k (k=1,3,5) of your retriever. Report results and discuss failures.

3. **Prompt engineering**: Create two different final-answer prompts (concise vs. verbose). Compare LLM outputs for 5 queries and state which prompt performs better and why.

4. **Embeddings comparison**: Embed the same set of chunks with two different embedding models and compare nearest neighbor overlap (top-5) for 20 queries. Summarize differences.

5. **Adversarial queries**: Design 5 adversarial questions that aim to produce hallucinations from RAG (e.g., ambiguous, missing info). Show how to mitigate them (retriever tuning, prompt changes, provenance).

6. **Cost analysis**: Estimate API token and embedding costs for running your pipeline on 1,000 queries per month. Suggest two cost-saving strategies.

7. **Extension (optional, challenge)**: Integrate a simple re-ranker (cross-encoder or lightweight model) and show improvement on one example query with before/after outputs.